# 02 — Validación Bronze → Silver

Valida el renombrado canónico, tipos, timestamps UTC, DQ/Quarantine, trazabilidad y clasificación de torque. La validación sobre un único archivo utiliza carpetas temporales de salida, por lo que **no puede sobrescribir Silver oficial**. La última celda ejecuta el pipeline batch incremental real para todos los archivos Bronze.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
print('Raíz del proyecto:', PROJECT_ROOT)

In [ ]:
from tempfile import TemporaryDirectory
from schemas.silver_schema import COLUMN_MAPPING, SILVER_SCHEMA_VERSION
from transformation.bronze_to_silver import transform_bronze_to_silver
from pipelines.run_batch_pipeline import run_batch_pipeline

BRONZE_ROOT = PROJECT_ROOT / 'data' / 'bronze'
files = sorted(BRONZE_ROOT.glob('*.csv'))
if not files:
    raise FileNotFoundError(BRONZE_ROOT)
BRONZE_FILE = files[0]
print('Schema Silver:', SILVER_SCHEMA_VERSION)
print('Fuente de validación:', BRONZE_FILE.name)

## Mapping canónico de origen

In [ ]:
mapping = pd.DataFrame(COLUMN_MAPPING.items(), columns=['bronze_field','silver_field'])
display(mapping)

## Transformación aislada de un archivo

In [ ]:
with TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    silver, quarantine = transform_bronze_to_silver(
        bronze_path=BRONZE_FILE,
        silver_root=tmp/'silver',
        quarantine_root=tmp/'quarantine',
    )

print('Filas Silver:', len(silver))
print('Filas Quarantine:', len(quarantine))
display(silver.head())

## Comprobaciones del contrato Silver

In [ ]:
expected_columns = set(COLUMN_MAPPING.values()) | {
    'source_row_number','source_file','ingestion_timestamp',
    'torque_applicable','torque_spec_status','dq_is_valid','dq_error_codes',
}
missing = sorted(expected_columns - set(silver.columns))
assert not missing, missing
assert isinstance(silver['event_timestamp'].dtype, pd.DatetimeTZDtype)
assert str(silver['event_timestamp'].dt.tz) == 'UTC'
assert set(silver['torque_spec_status'].dropna().unique()).issubset({'IN_SPEC','OUT_OF_SPEC','NOT_APPLICABLE'})
assert {'factory_id','tightening_unit_id','process_step_id','process_step_name','substep_id','substep_number'}.issubset(silver.columns)
assert not {'component_id','component_number','component_context','controller_id','controller_name','controller_ip','product_trace_id','result_number','result_substep_number','error_code','stop_source','cycle_ok_count'}.intersection(silver.columns)
assert len(COLUMN_MAPPING) == 24
print('Contrato semántico y de tipos Silver superado (24 campos de negocio de origen).')

## Reconciliación de DQ y torque

In [ ]:
assert len(silver) + len(quarantine) > 0
applicable = int(silver['torque_applicable'].sum())
in_spec = int((silver['torque_spec_status'] == 'IN_SPEC').sum())
out_spec = int((silver['torque_spec_status'] == 'OUT_OF_SPEC').sum())
not_applicable = int((silver['torque_spec_status'] == 'NOT_APPLICABLE').sum())
assert applicable == in_spec + out_spec
assert len(silver) == applicable + not_applicable
print({'applicable': applicable, 'in_spec': in_spec, 'out_of_spec': out_spec, 'not_applicable': not_applicable})

## Batch incremental oficial

Este es el camino operativo. Los archivos Silver ya existentes se omiten; las salidas Silver antiguas o incompatibles se reconstruyen automáticamente desde Bronze.

In [ ]:
batch_summary = run_batch_pipeline(
    bronze_root=PROJECT_ROOT/'data'/'bronze',
    silver_root=PROJECT_ROOT/'data'/'silver',
    quarantine_root=PROJECT_ROOT/'data'/'quarantine',
    force=False,
)
display(batch_summary)
assert not batch_summary['status'].eq('FAILED').any()
print('El batch oficial Bronze → Silver está listo.')